In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('Crashes.csv')

# View the first 5 rows of your data
df.head()

# Convert military time (HHMM) to an hour-of-day value
# Example: 2206 -> 22, 0136 -> 01
# This makes it possible to measure the most common crash time
# across the whole dataset.
df['MILT_TIME'] = df['MILT_TIME'].fillna('0000').astype(str).str.zfill(4)
df['HourOfDay'] = df['MILT_TIME'].str[:2].astype(int)

# Most likely time of day for crashes
hour_counts = df['HourOfDay'].value_counts().sort_index()
most_likely_hour = hour_counts.index[hour_counts.argmax()]
print(f"Most likely crash hour: {most_likely_hour:02d}:00")
print(hour_counts.head(10))

# Most likely year for crashes
year_counts = df['DATE_VAL_YEAR'].value_counts().sort_index()
most_likely_year = year_counts.index[year_counts.argmax()]
print(f"Most likely crash year: {most_likely_year}")
print(year_counts.head(10))

# Most likely day of the week for crashes
weekday_counts = df['DAY_OF_WEEK_DESC'].value_counts()
most_likely_weekday = weekday_counts.index[weekday_counts.argmax()]
print(f"Most likely day of week: {most_likely_weekday}")
print(weekday_counts.head(10))

# Most likely specific month of the year for crashes
month_counts = df['DATE_VAL_MONTH_DESC'].value_counts()
most_likely_month = month_counts.index[month_counts.argmax()]
print(f"Most likely crash month: {most_likely_month}")
print(month_counts.head(12))

In [6]:
# Analyze crashes around major holiday dates
# This looks at a 5-day window around each holiday (2 days before, the date itself, and
# 2 days after) and compares that to the average daily crash rate for the same year.

import pandas as pd

# Make a clean date column from the actual date fields in the CSV file.
# The source data uses DATE_VAL, DATE_VAL_YEAR, DATE_VAL_MONTH, and DATE_VAL_DAY.
if 'DATE_VAL' in df.columns:
    valid = df.copy()
    valid['CrashDate'] = pd.to_datetime(valid['DATE_VAL'], errors='coerce', utc=True).dt.tz_localize(None)
else:
    month_map = {
        'JANUARY': 1, 'FEBRUARY': 2, 'MARCH': 3, 'APRIL': 4, 'MAY': 5, 'JUNE': 6,
        'JULY': 7, 'AUGUST': 8, 'SEPTEMBER': 9, 'OCTOBER': 10, 'NOVEMBER': 11, 'DECEMBER': 12
    }
    if 'DATE_VAL_MONTH_DESC' in df.columns:
        df['DATE_VAL_MONTH'] = df['DATE_VAL_MONTH_DESC'].str.upper().map(month_map)
    if 'DATE_VAL_DAY_OF_MONTH' in df.columns:
        day_col = 'DATE_VAL_DAY_OF_MONTH'
    else:
        day_col = 'DATE_VAL_DAY'
    df['DATE_VAL_MONTH'] = pd.to_numeric(df['DATE_VAL_MONTH'], errors='coerce')
    df[day_col] = pd.to_numeric(df[day_col], errors='coerce')
    df['DATE_VAL_YEAR'] = pd.to_numeric(df['DATE_VAL_YEAR'], errors='coerce')

    valid = df.dropna(subset=['DATE_VAL_YEAR', 'DATE_VAL_MONTH', day_col]).copy()
    valid['CrashDate'] = pd.to_datetime(
        valid['DATE_VAL_YEAR'].astype(int).astype(str) + '-' +
        valid['DATE_VAL_MONTH'].astype(int).astype(str).str.zfill(2) + '-' +
        valid[day_col].astype(int).astype(str).str.zfill(2),
        errors='coerce', utc=True
    ).dt.tz_localize(None)

valid = valid.dropna(subset=['CrashDate']).copy()

# Major dates to analyze
holiday_map = {
    "New Year's Day": (1, 1),
    "Thanksgiving": "THANKSGIVING",
    "Independence Day": (7, 4),
    "Halloween": (10, 31),
    "Veterans Day": (11, 11),
    "Christmas": (12, 25),
}

holiday_results = []

for holiday_name, holiday_spec in holiday_map.items():
    for year in sorted(valid['DATE_VAL_YEAR'].dropna().unique().astype(int)):
        year_df = valid[valid['DATE_VAL_YEAR'] == year].copy()
        if year_df.empty:
            continue

        year_start = pd.Timestamp(f'{year}-01-01')
        year_end = pd.Timestamp(f'{year}-12-31')
        total_days = len(pd.date_range(year_start, year_end, freq='D'))
        avg_daily_crashes = len(year_df) / total_days

        if holiday_spec == 'THANKSGIVING':
            thanksgiving = pd.Timestamp(year, 11, 1)
            thanksgiving = thanksgiving + pd.to_timedelta((3 - thanksgiving.weekday()) % 7, unit='D')
            thanksgiving = thanksgiving + pd.to_timedelta(21, unit='D')
            target_date = thanksgiving
        else:
            month, day = holiday_spec
            target_date = pd.Timestamp(f'{year}-{month:02d}-{day:02d}')

        window_start = target_date - pd.Timedelta(days=2)
        window_end = target_date + pd.Timedelta(days=2)

        window_df = year_df[(year_df['CrashDate'] >= window_start) & (year_df['CrashDate'] <= window_end)]
        window_total = len(window_df)
        window_avg_daily = window_total / 5

        holiday_results.append({
            'Holiday': holiday_name,
            'Year': year,
            'TargetDate': target_date.strftime('%Y-%m-%d'),
            'CrashCountInWindow': window_total,
            'AverageDailyCrashes': round(avg_daily_crashes, 2),
            'WindowAvgDailyCrashes': round(window_avg_daily, 2),
            'RateVsAverage': round(window_avg_daily / avg_daily_crashes, 2) if avg_daily_crashes else None,
        })

holiday_df = pd.DataFrame(holiday_results)

if not holiday_df.empty:
    summary = (
        holiday_df.groupby('Holiday')
        .agg(
            TotalCrashes=('CrashCountInWindow', 'sum'),
            AvgDailyInWindow=('WindowAvgDailyCrashes', 'mean'),
            AvgDailyOverall=('AverageDailyCrashes', 'mean'),
            RateMultiplier=('RateVsAverage', 'mean')
        )
        .reset_index()
        .sort_values('TotalCrashes', ascending=False)
    )

    print('Crash rates around major dates (±2 days)')
    print(summary[['Holiday', 'TotalCrashes', 'AvgDailyInWindow', 'AvgDailyOverall', 'RateMultiplier']].to_string(index=False))
    print('\n')

    for holiday_name in holiday_map:
        subset = holiday_df[holiday_df['Holiday'] == holiday_name]
        print(f"{holiday_name}")
        print(subset[['Year', 'TargetDate', 'CrashCountInWindow', 'AverageDailyCrashes', 'WindowAvgDailyCrashes', 'RateVsAverage']].to_string(index=False))
        print('')
else:
    print('No valid date data was found for the major-date analysis.')

Crash rates around major dates (±2 days)
         Holiday  TotalCrashes  AvgDailyInWindow  AvgDailyOverall  RateMultiplier
       Halloween          6391           79.8875        85.224375        0.900000
    Veterans Day          5855           73.1875        85.224375        0.811875
    Thanksgiving          5251           65.6375        85.224375        0.773750
       Christmas          4317           53.9625        85.224375        0.603750
Independence Day          4023           50.2875        85.224375        0.545000
  New Year's Day          2158           26.9750        85.224375        0.346875


New Year's Day
 Year TargetDate  CrashCountInWindow  AverageDailyCrashes  WindowAvgDailyCrashes  RateVsAverage
 2010 2010-01-01                  65                42.82                   13.0           0.30
 2011 2011-01-01                 108                42.19                   21.6           0.51
 2012 2012-01-01                  48                40.22                    9.6

In [7]:
from scipy.stats import poisson

holiday_stat_rows = []

for holiday_name, holiday_spec in holiday_map.items():
    subset = holiday_df[holiday_df['Holiday'] == holiday_name].copy()
    if subset.empty:
        continue

    for _, row in subset.iterrows():
        observed = int(row['CrashCountInWindow'])
        expected_daily = float(row['AverageDailyCrashes'])
        expected_window = expected_daily * 5
        rate_ratio = float(row['RateVsAverage']) if pd.notna(row['RateVsAverage']) else float('nan')
        p_value = float(poisson.sf(observed - 1, expected_window)) if expected_window > 0 else 1.0

        holiday_stat_rows.append({
            'Holiday': holiday_name,
            'Year': int(row['Year']),
            'ObservedCrashes': observed,
            'ExpectedCrashes5DayWindow': round(expected_window, 2),
            'RateRatioVsYearAvg': round(rate_ratio, 2),
            'p_value': p_value,
            'SignificantAt05': p_value < 0.05,
            'SignificantAt01': p_value < 0.01,
        })

holiday_stats = pd.DataFrame(holiday_stat_rows)

if not holiday_stats.empty:
    summary_by_holiday = (
        holiday_stats.groupby('Holiday')
        .agg(
            Years=('Year', 'count'),
            AvgRateRatio=('RateRatioVsYearAvg', 'mean'),
            StrongestRateRatio=('RateRatioVsYearAvg', 'max'),
            LowestPValue=('p_value', 'min'),
            SignificantYears05=('SignificantAt05', 'sum'),
            SignificantYears01=('SignificantAt01', 'sum')
        )
        .reset_index()
        .sort_values(['AvgRateRatio', 'LowestPValue'], ascending=[False, True])
    )
    print('Holiday significance vs same-year average (one-sided Poisson test)')
    print(summary_by_holiday[['Holiday', 'Years', 'AvgRateRatio', 'StrongestRateRatio', 'LowestPValue', 'SignificantYears05', 'SignificantYears01']].to_string(index=False))
    print('\n')

    print('Per-holiday, per-year significance results')
    print(holiday_stats.sort_values(['Holiday', 'Year']).to_string(index=False))
else:
    print('No holiday statistics were produced.')


Holiday significance vs same-year average (one-sided Poisson test)
         Holiday  Years  AvgRateRatio  StrongestRateRatio  LowestPValue  SignificantYears05  SignificantYears01
       Halloween     16      0.900000                1.23      0.000041                   2                   2
    Veterans Day     16      0.811875                1.21      0.000001                   2                   1
    Thanksgiving     16      0.773750                1.11      0.056764                   0                   0
       Christmas     16      0.603750                0.87      0.971774                   0                   0
Independence Day     16      0.545000                0.77      1.000000                   0                   0
  New Year's Day     16      0.346875                0.97      0.679648                   0                   0


Per-holiday, per-year significance results
         Holiday  Year  ObservedCrashes  ExpectedCrashes5DayWindow  RateRatioVsYearAvg  p_value  Signifi

In [8]:
from scipy.stats import poisson

pandemic_years = [2020, 2021, 2022]
all_years = sorted(valid['DATE_VAL_YEAR'].dropna().unique().astype(int))
non_pandemic_years = [year for year in all_years if year not in pandemic_years]

# Baseline rate from non-pandemic years
non_pandemic_rows = []
non_pandemic_total_crashes = 0
non_pandemic_total_days = 0

for year in non_pandemic_years:
    year_df = valid[valid['DATE_VAL_YEAR'] == year].copy()
    days = len(pd.date_range(pd.Timestamp(f'{year}-01-01'), pd.Timestamp(f'{year}-12-31'), freq='D'))
    observed = len(year_df)
    non_pandemic_total_crashes += observed
    non_pandemic_total_days += days
    non_pandemic_rows.append({
        'Year': year,
        'ObservedCrashes': observed,
        'Days': days,
        'AvgDailyCrashes': observed / days,
        'PandemicPeriod': False,
    })

non_pandemic_daily_rate = non_pandemic_total_crashes / non_pandemic_total_days

# Pandemic-year comparison against the non-pandemic baseline
pandemic_rows = []
pandemic_total_crashes = 0
pandemic_total_days = 0

for year in pandemic_years:
    year_df = valid[valid['DATE_VAL_YEAR'] == year].copy()
    days = len(pd.date_range(pd.Timestamp(f'{year}-01-01'), pd.Timestamp(f'{year}-12-31'), freq='D'))
    observed = len(year_df)
    expected = non_pandemic_daily_rate * days
    p_value = float(poisson.cdf(observed, expected))
    pandemic_rows.append({
        'Year': year,
        'ObservedCrashes': observed,
        'ExpectedCrashes': round(expected, 2),
        'AvgDailyCrashes': observed / days,
        'RateRatioVsBaseline': round((observed / days) / non_pandemic_daily_rate, 2),
        'p_value': p_value,
        'LowerThanBaseline': p_value < 0.05,
        'PandemicPeriod': True,
    })
    pandemic_total_crashes += observed
    pandemic_total_days += days

expected_pandemic_total = non_pandemic_daily_rate * pandemic_total_days
pandemic_p_value = float(poisson.cdf(pandemic_total_crashes, expected_pandemic_total))
pandemic_rate_ratio = pandemic_total_crashes / expected_pandemic_total

print('Pandemic (2020-2022) vs non-pandemic baseline')
print(f"Non-pandemic baseline daily crashes: {non_pandemic_daily_rate:.4f}")
print(f"Pandemic total crashes: {pandemic_total_crashes} vs expected {expected_pandemic_total:.2f}")
print(f"Pandemic rate ratio vs baseline: {pandemic_rate_ratio:.4f}")
print(f"One-sided p-value for a decrease: {pandemic_p_value:.10f}")
print(f"Statistically lower than baseline at alpha=0.05: {pandemic_p_value < 0.05}")
print('\n')
print('Per-year pandemic comparison')
print(pd.DataFrame(pandemic_rows)[['Year', 'ObservedCrashes', 'ExpectedCrashes', 'AvgDailyCrashes', 'RateRatioVsBaseline', 'p_value', 'LowerThanBaseline']].to_string(index=False))
print('\n')
print('Non-pandemic baseline years summary')
print(pd.DataFrame(non_pandemic_rows)[['Year', 'ObservedCrashes', 'Days', 'AvgDailyCrashes']].to_string(index=False))


Pandemic (2020-2022) vs non-pandemic baseline
Non-pandemic baseline daily crashes: 81.2734
Pandemic total crashes: 112173 vs expected 89075.62
Pandemic rate ratio vs baseline: 1.2593
One-sided p-value for a decrease: 1.0000000000
Statistically lower than baseline at alpha=0.05: False


Per-year pandemic comparison
 Year  ObservedCrashes  ExpectedCrashes  AvgDailyCrashes  RateRatioVsBaseline  p_value  LowerThanBaseline
 2020            29345         29746.06        80.177596                 0.99  0.00999               True
 2021            38560         29664.78       105.643836                 1.30  1.00000              False
 2022            44268         29664.78       121.282192                 1.49  1.00000              False


Non-pandemic baseline years summary
 Year  ObservedCrashes  Days  AvgDailyCrashes
 2010            15631   365        42.824658
 2011            15399   365        42.189041
 2012            14722   366        40.224044
 2013            23211   365        63

In [10]:
# Monthly crash analysis including fatal crashes
# Build a month-level summary using the same valid crash date data already created above.

if 'CrashDate' in valid.columns:
    valid['IsFatal'] = valid['CRSH_LEVL_DESC'].fillna('').str.lower().str.contains('fatal', na=False)

    month_order = [
        'JANUARY', 'FEBRUARY', 'MARCH', 'APRIL', 'MAY', 'JUNE',
        'JULY', 'AUGUST', 'SEPTEMBER', 'OCTOBER', 'NOVEMBER', 'DECEMBER'
    ]
    month_lookup = {name: i for i, name in enumerate(month_order, start=1)}

    monthly_summary = (
        valid.assign(
            MonthNum=valid['DATE_VAL_MONTH'].astype('Int64'),
            MonthName=valid['DATE_VAL_MONTH_DESC'].fillna('').str.upper()
        )
        .groupby('MonthNum', dropna=False)
        .agg(
            TotalCrashes=('CrashDate', 'size'),
            FatalCrashes=('IsFatal', 'sum')
        )
        .reset_index()
        .assign(
            MonthName=lambda df: df['MonthNum'].map({v: k for k, v in month_lookup.items()})
        )
    )

    monthly_summary['FatalSharePct'] = (monthly_summary['FatalCrashes'] / monthly_summary['TotalCrashes'] * 100).round(2)
    monthly_summary = monthly_summary[['MonthName', 'TotalCrashes', 'FatalCrashes', 'FatalSharePct']].sort_values('MonthName', key=lambda s: s.map(month_lookup)).reset_index(drop=True)

    print('Monthly crash totals and fatal crashes')
    print(monthly_summary.to_string(index=False))
    print('\n')

    peak_month = monthly_summary.loc[monthly_summary['TotalCrashes'].idxmax()]
    peak_fatal_month = monthly_summary.loc[monthly_summary['FatalCrashes'].idxmax()]
    print(f"Month with the highest total crashes: {peak_month['MonthName']} ({peak_month['TotalCrashes']} crashes)")
    print(f"Month with the highest fatal crashes: {peak_fatal_month['MonthName']} ({peak_fatal_month['FatalCrashes']} fatal crashes)")
else:
    print('No valid crash date column is available for the monthly analysis.')


Monthly crash totals and fatal crashes
MonthName  TotalCrashes  FatalCrashes  FatalSharePct
  JANUARY         42232            93           0.22
 FEBRUARY         37878            78           0.21
    MARCH         43666            93           0.21
    APRIL         40849            76           0.19
      MAY         41985            93           0.22
     JUNE         37849            82           0.22
     JULY         37365            76           0.20
   AUGUST         41192           102           0.25
SEPTEMBER         41730            84           0.20
  OCTOBER         46484           101           0.22
 NOVEMBER         43540            72           0.17
 DECEMBER         43289            93           0.21


Month with the highest total crashes: OCTOBER (46484 crashes)
Month with the highest fatal crashes: AUGUST (102 fatal crashes)
